# lidarMONK · 3D-Rekonstruktion in Google Colab

Mit zwei oder drei Aufnahmen desselben Motivs kannst du das experimentelle Backend in einer kostenlosen Colab-**CPU**-Sitzung ausprobieren. Zellen nacheinander mit ▶ ausführen.

**Dateien:** WurldCam `.webm`/`.mkv` oder vollständiges Stray-Scanner- bzw. SplatCapture-Export-ZIP. Ein ZIP muss genau einen vollständigen Export enthalten. SplatCapture wird derzeit nur als Punktwolke eingelesen. Gewöhnliche Fotos oder Videos ohne passende Tiefen- und Posedaten funktionieren nicht.

**Vor dem Upload:** Deine Dateien werden zur Verarbeitung an Google Colab übertragen und liegen vorübergehend auf dessen virtueller Maschine. Benutze nur Scans, die du dort verarbeiten möchtest. Keine Scan-Dateien oder Zellenausgaben im öffentlichen GitHub-Repository speichern. Die Maschine und ihre Dateien können ohne Vorwarnung verschwinden; Ergebnis sofort herunterladen.

**Grenzen:** Für kleine Tests gedacht. Colab läuft nicht ständig, Verfügbarkeit und Speicher sind nicht garantiert. Die mobile Bedienung in Safari ist nicht überprüft. Registrierung, Geometrie und Personenkorrektur sind experimentell; keine Messgenauigkeit zugesichert. Falsche Ausrichtung ist möglich.

## 1 · Backend vorbereiten
Die erste Ausführung installiert Abhängigkeiten. Das dauert einige Minuten.

In [ ]:
import os, sys, subprocess, urllib.request
from pathlib import Path

ROOT = Path('/content/lidarmonk-colab')
ROOT.mkdir(parents=True, exist_ok=True)
BACKEND = ROOT / 'backend'
BACKEND.mkdir(exist_ok=True)
# Fester geprüfter Repository-Stand, kein automatisches Ausführen wechselnder main-Inhalte.
REVISION = 'fa771abb8866196121b8aa60cd0893172345b107'
BASE = f'https://raw.githubusercontent.com/kAInplanmusic/lidarMONK/{REVISION}/scan-lab/backend/'
for name in ['server.py', 'lidarmonk_backend.py', 'motion.py', 'regional_tsdf.py', 'requirements.txt']:
    urllib.request.urlretrieve(BASE + name, BACKEND / name)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BACKEND / 'requirements.txt')], check=True)
subprocess.run(['ffmpeg', '-version'], stdout=subprocess.DEVNULL, check=True)
os.environ['OMP_NUM_THREADS'] = '2'
sys.path.insert(0, str(BACKEND))
print('Bereit. Jetzt die nächste Zelle ausführen.')

## 2 · Zwei oder drei Aufnahmen hochladen
Wähle **alle Dateien gleichzeitig** aus. Bei großen Aufnahmen kann der Upload viel Arbeitsspeicher belegen oder auf dem iPhone abbrechen. Für den ersten Test kleine Aufnahmen verwenden.

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

uploaded = files.upload()
if not 2 <= len(uploaded) <= 3:
    raise ValueError('Bitte genau zwei oder drei unterschiedliche Dateien auswählen und die Zelle erneut ausführen.')
INPUT = ROOT / 'input'
if INPUT.exists():
    shutil.rmtree(INPUT)
INPUT.mkdir()
captures = []
for index, (name, content) in enumerate(uploaded.items(), start=1):
    suffix = Path(name).suffix.lower()
    if suffix not in ('.zip', '.webm', '.mkv'):
        raise ValueError(f'{name}: nur ZIP, WEBM und MKV werden unterstützt.')
    path = INPUT / f'aufnahme-{index}{suffix}'
    path.write_bytes(content)
    captures.append(path)
del uploaded
print(f'{len(captures)} Dateien bereit.')

## 3 · Modell berechnen
Die automatische Ausrichtung benötigt ausreichend Überlappung zwischen den Aufnahmen. Wenn sie abbricht, verwende mehr gemeinsame Ansichten. Personen sollten möglichst stillhalten.

In [ ]:
import json, subprocess, sys, zipfile
from datetime import datetime
from server import unzip_checked

# Optionen: True schaltet die jeweilige experimentelle Funktion ein.
BEWEGUNGSKORREKTUR = False
REGIONALES_TSDF = True
# Nur einschalten, wenn Tiefenbild und Farbkamera nachweislich denselben Bildausschnitt zeigen.
GLEICHER_BILDAUSSCHNITT = False

prepared = []
for index, path in enumerate(captures, start=1):
    if path.suffix == '.zip':
        destination = ROOT / f'entpackt-{index}'
        if destination.exists():
            shutil.rmtree(destination)
        destination.mkdir()
        prepared.append(unzip_checked(path, destination))
    else:
        prepared.append(path)
# Jeder Lauf erhält einen eigenen Ergebnisordner; vorhandene Ergebnisse werden nicht überschrieben.
output = ROOT / ('ergebnis-' + datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
cmd = [sys.executable, str(BACKEND / 'lidarmonk_backend.py'), str(prepared[0]), str(prepared[1]), '--output', str(output), '--voxel', '0.08']
if len(prepared) == 3:
    cmd += ['--capture-c', str(prepared[2])]
if BEWEGUNGSKORREKTUR:
    cmd += ['--motion-correction']
if REGIONALES_TSDF:
    cmd += ['--regional-tsdf']
if GLEICHER_BILDAUSSCHNITT:
    cmd += ['--aligned-depth']
print('Berechnung läuft …')
subprocess.run(cmd, check=True)
archive = shutil.make_archive(str(output), 'zip', root_dir=output)
print('Fertig. Ergebnisdateien:', ', '.join(p.name for p in output.iterdir() if p.is_file()))
print('Jetzt die Download-Zelle ausführen.')

## 4 · Ergebnisse herunterladen
Das ZIP enthält die erzeugten Punktwolken und Berichte. Modelle und Ausrichtung visuell prüfen; die Ergebnisse können unvollständig oder fehlerhaft sein.

In [ ]:
from google.colab import files
files.download(archive)